# Smart Traffic Model Training (Google Colab)

Train the Bangalore traffic model here, export the artifact, and copy it back into `backend/saved_models/` on localhost.

In [ ]:
!pip install pandas scikit-learn xgboost joblib

In [ ]:
import joblib
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

DATA_PATH = Path('/content/Banglore_traffic_Dataset.csv')
df = pd.read_csv(DATA_PATH)
df['Date'] = pd.to_datetime(df['Date'])
df['hour'] = df['Date'].dt.hour
df['day_of_week'] = df['Date'].dt.dayofweek
df['month'] = df['Date'].dt.month
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
df = df.dropna().copy()

feature_columns = [
    'Area Name', 'Road/Intersection Name', 'Weather Conditions',
    'hour', 'day_of_week', 'month', 'is_weekend',
    'Average Speed', 'Travel Time Index', 'Road Capacity Utilization',
    'Incident Reports', 'Public Transport Usage', 'Pedestrian and Cyclist Count'
]
target_column = 'Traffic Volume'

X = df[feature_columns]
y = df[target_column]

categorical = ['Area Name', 'Road/Intersection Name', 'Weather Conditions']
numeric = [column for column in feature_columns if column not in categorical]

preprocessor = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical),
    ('numeric', 'passthrough', numeric),
])

model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        objective='reg:squarederror',
        random_state=42,
    ))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = mean_squared_error(y_test, predictions, squared=False)
mape = ((y_test - predictions).abs() / y_test.clip(lower=1)).mean() * 100
print({'MAE': mae, 'RMSE': rmse, 'MAPE': mape})

joblib.dump(model, '/content/xgboost_model.pkl')
print('Saved /content/xgboost_model.pkl')

Download `xgboost_model.pkl` from Colab and place it into `backend/saved_models/` on localhost.